# Kaggle vLLM + ngrok

Run cells in order. GPU: T4 x2.

Requires Kaggle secrets: `NGROK_AUTH_TOKEN`, `HF_TOKEN`.

When done, opencode.json `baseURL` = printed NGROK URL + `/v1`.

Speed notes: T4s have no NVLink (PHB topology), so default `TP=1` (single GPU) — faster than TP=2 due to zero PCIe sync per token. Set `VLLM_TP_SIZE=2` only if you need 2x KV cache for long contexts. `--enforce-eager` removed — CUDA graphs speed up generation.

In [1]:
%pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 95.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 103.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 44.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
!pkill -9 -x ngrok || true
!sleep 1
!which ngrok || curl -sSL -o /tmp/ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -q -o /tmp/ngrok.zip -d /tmp || true
!mv -f /tmp/ngrok /usr/local/bin/ngrok || true

In [3]:
!pkill -9 -if vllm || true
!pkill -9 -x ngrok || true
!fuser -k 8000/tcp 2>/dev/null || true
!sleep 2
!nvidia-smi

Fri Aug 28 08:32:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import importlib
import json
import os
import subprocess
import time
import urllib.error
import urllib.request

try:
    kaggle_secrets = importlib.import_module("kaggle_secrets")
    UserSecretsClient = kaggle_secrets.UserSecretsClient
except ModuleNotFoundError as exc:
    raise RuntimeError("This cell must be run in a Kaggle notebook.") from exc

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

subprocess.run(
    ["ngrok", "config", "add-authtoken", secrets.get_secret("NGROK_AUTH_TOKEN")],
    check=True,
)
with open("/tmp/ngrok.log", "w") as ngrok_log:
    p = subprocess.Popen(
        ["ngrok", "http", "8000", "--log", "stdout"],
        stdout=ngrok_log,
        stderr=subprocess.STDOUT,
    )

for _ in range(60):
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:4040/api/tunnels", timeout=5
        ) as response:
            d = json.load(response)
        url = d["tunnels"][0]["public_url"]
        print("NGROK URL:", url)
        break
    except (urllib.error.URLError, json.JSONDecodeError, KeyError, IndexError):
        time.sleep(2)
else:
    with open("/tmp/ngrok.log") as ngrok_log:
        print(ngrok_log.read())

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
NGROK URL: https://5e3c-34-48-41-101.ngrok-free.app


In [ ]:
MODEL = os.environ.get("VLLM_MODEL", "Qwen/Qwen2.5-7B-Instruct-AWQ")
TP = int(os.environ.get("VLLM_TP_SIZE", "1"))

subprocess.run(
    [
        "vllm",
        "serve",
        MODEL,
        "--served-model-name",
        MODEL,
        "--tensor-parallel-size",
        str(TP),
        "--max-model-len",
        "16384",
        "--gpu-memory-utilization",
        "0.90",
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
        "--port",
        "8000",
    ],
    check=True,
)

(APIServer pid=255) INFO 08-28 08:32:53 [api_utils.py:333] 
(APIServer pid=255) INFO 08-28 08:32:53 [api_utils.py:333]        █     █     █▄   ▄█
(APIServer pid=255) INFO 08-28 08:32:53 [api_utils.py:333]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.28.0
(APIServer pid=255) INFO 08-28 08:32:53 [api_utils.py:333]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-7B-Instruct-AWQ
(APIServer pid=255) INFO 08-28 08:32:53 [api_utils.py:333]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=255) INFO 08-28 08:32:53 [api_utils.py:333] 
(APIServer pid=255) INFO 08-28 08:32:53 [api_utils.py:272] non-default args: {'model_tag': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'enable_auto_tool_choice': True, 'tool_call_parser': 'hermes', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'max_model_len': 16384, 'served_model_name': ['Qwen/Qwen2.5-7B-Instruct-AWQ'], 'gpu_memory_utilization': 0.9}
(APIServer pid=255) INFO 08-28 08:33:09 [model.py:672] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=255) INFO 08-28 08:33:09 [model.py:

Parse safetensors files: 100%|██████████| 2/2 [00:00<00:00,  5.68it/s]


(APIServer pid=255) INFO 08-28 08:33:10 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=330) INFO 08-28 08:33:31 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=auto_awq, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_addit

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.27s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.17it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.09it/s]
(EngineCore pid=330) 


(EngineCore pid=330) INFO 08-28 08:34:01 [default_loader.py:430] Loading weights took 1.87 seconds
(EngineCore pid=330) INFO 08-28 08:34:04 [model_runner.py:380] Model loading took 5.29 GiB memory and 31.473666 seconds
(EngineCore pid=330) WARNING 08-28 08:34:04 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=330) INFO 08-28 08:34:17 [backends.py:1094] Using cache directory: /root/.cache/vllm/torch_compile_cache/2e37374793/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=330) INFO 08-28 08:34:17 [backends.py:1155] Dynamo bytecode transform time: 10.30 s
(EngineCore pid=330) INFO 08-28 08:34:27 [backends.py:393] Compiling a graph for compile range (1, 2048) takes 9.65 s
(EngineCore pid=330) INFO 08-28 08:34:32 [backends.py:920] collected artifacts: 29 entries, 3 artifacts, 4656877 bytes total
(EngineCore pid=330) INFO 08-28 08:34:32 [decorators

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:03<00:00,  9.00it/s]


(EngineCore pid=330) INFO 08-28 08:34:54 [model_runner.py:906] Graph capturing finished in 18 secs, took 0.51 GiB
(EngineCore pid=330) INFO 08-28 08:34:54 [gpu_worker.py:804] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 6.58 GiB for consumed memory (weights + non-torch), 1.02 GiB for peak activation, and 0.51 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=5203431629` (4.85 GiB) to fit into requested memory, or `--kv-cache-memory=6657065984` (6.2 GiB) to fully utilize gpu memory. Current kv cache memory in use is 5.51 GiB.
(EngineCore pid=330) INFO 08-28 08:35:03 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=330) INFO 08-28 08:35:04 [torch_utils.py:262] Reducing Torch threads from 2 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=330) INFO 08-2

(APIServer pid=255) INFO:     Started server process [255]
(APIServer pid=255) INFO:     Waiting for application startup.
(APIServer pid=255) INFO:     Application startup complete.


(APIServer pid=255) INFO:     42.117.147.62:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=255) INFO:     42.117.147.62:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(EngineCore pid=330) WARNING 08-28 08:44:43 [jit_monitor.py:141] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(APIServer pid=255) INFO 08-28 08:44:47 [loggers.py:310] Engine 000: Avg prompt throughput: 289.5 tokens/s, Avg generation throughput: 5.8 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 1.4%, Prefix cache hit rate: 0.0%
(APIServer pid=255) INFO 08-28 08:44:57 [loggers.py:310] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 39.1 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 1.8%, Prefix cache hit rate: 0.0%
(APIServer pid=255) INFO:     42.117.147.62:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=255) INFO:    